# **Carga y Gestión de Datos con SQLite**
# **Airbnb Listings — Ciudad de México**

Este notebook cubre el punto **1** del Laboratorio #3: creación de la base de datos SQLite, importación del dataset de regresión y ejecución de consultas SQL relevantes.

**Curso**: Analítica de Datos — Universidad de Antioquia  
**Profesor**: Duván Cataño  
**Laboratorio #3**

---
# 1. Librerías

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

---
# 2. Creación de la Base de Datos SQLite

In [ ]:
# Crear carpeta database/ si no existe
os.makedirs('../database', exist_ok=True)

# Ruta de la base de datos
DB_PATH = '../database/airbnb_listings.db'

# Crear conexión (crea el archivo .db automáticamente si no existe)
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

print(f'Base de datos creada/conectada exitosamente en: {DB_PATH}')

---
# 3. Importar el Dataset y Crear Tabla en SQLite

In [ ]:
# Cargar el CSV
filepath = '../data/dataset_regresion_listings.csv'
data = pd.read_csv(filepath, sep=',')

print(f'Dataset cargado: {data.shape[0]:,} filas × {data.shape[1]} columnas')
data.head(5)

In [ ]:
# Limpiar columnas 100% vacías antes de importar
# neighbourhood_group y license son 100% nulos en este dataset (CDMX)
data_db = data.drop(columns=['neighbourhood_group', 'license'])

# Convertir last_review a string limpio (SQLite no tiene tipo DATE nativo)
data_db['last_review'] = data_db['last_review'].astype(str).replace('nan', None)

print('Columnas que se importarán a SQLite:')
print(data_db.columns.tolist())

In [ ]:
# Crear tabla en SQLite con esquema explícito
cursor.execute('DROP TABLE IF EXISTS listings')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS listings (
        id                              INTEGER PRIMARY KEY,
        name                            TEXT,
        host_id                         INTEGER,
        host_name                       TEXT,
        neighbourhood                   TEXT,
        latitude                        REAL,
        longitude                       REAL,
        room_type                       TEXT,
        price                           REAL,
        minimum_nights                  INTEGER,
        number_of_reviews               INTEGER,
        last_review                     TEXT,
        reviews_per_month               REAL,
        calculated_host_listings_count  INTEGER,
        availability_365                INTEGER,
        number_of_reviews_ltm           INTEGER
    )
''')

conn.commit()
print('Tabla "listings" creada exitosamente.')

In [ ]:
# Importar datos usando pandas (método más eficiente)
data_db.to_sql('listings', conn, if_exists='replace', index=False)
conn.commit()

# Verificar carga
resultado = pd.read_sql('SELECT COUNT(*) AS total_registros FROM listings', conn)
print(f'Registros importados a SQLite: {resultado["total_registros"][0]:,}')

In [ ]:
# Verificar estructura de la tabla
print('Estructura de la tabla listings:')
info_tabla = pd.read_sql('PRAGMA table_info(listings)', conn)
print(info_tabla[['name', 'type', 'notnull', 'dflt_value']].to_string(index=False))

In [ ]:
# Vista previa desde SQLite
print('Primeras 5 filas desde SQLite:')
pd.read_sql('SELECT * FROM listings LIMIT 5', conn)

---
# 4. Consultas SQL Relevantes

A continuación se presentan **8 consultas SQL** que cubren todos los tipos requeridos: conteos, valores promedio, agrupaciones y filtros.

## 📊 Consulta 1 — Conteo total y por tipo de habitación
**Tipo:** Conteo  
**Objetivo:** Conocer cuántos listings existen en total y cómo se distribuyen por tipo de habitación.

In [ ]:
query_1 = '''
    SELECT
        room_type,
        COUNT(*)                              AS total_listings,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM listings), 2) AS porcentaje
    FROM listings
    GROUP BY room_type
    ORDER BY total_listings DESC
'''

resultado_1 = pd.read_sql(query_1, conn)
print('Conteo de listings por tipo de habitación:')
print(resultado_1.to_string(index=False))

total = pd.read_sql('SELECT COUNT(*) AS total FROM listings', conn)
print(f'\nTotal de listings en la base de datos: {total["total"][0]:,}')

## 💰 Consulta 2 — Precio promedio, mediano y máximo por tipo de habitación
**Tipo:** Valores promedio  
**Objetivo:** Comparar las estadísticas de precio según el tipo de alojamiento ofrecido.

In [ ]:
query_2 = '''
    SELECT
        room_type,
        COUNT(price)                  AS listings_con_precio,
        ROUND(AVG(price), 2)          AS precio_promedio,
        ROUND(MIN(price), 2)          AS precio_minimo,
        ROUND(MAX(price), 2)          AS precio_maximo,
        ROUND(AVG(minimum_nights), 2) AS noches_minimas_promedio
    FROM listings
    WHERE price IS NOT NULL
    GROUP BY room_type
    ORDER BY precio_promedio DESC
'''

resultado_2 = pd.read_sql(query_2, conn)
print('Estadísticas de precio por tipo de habitación:')
print(resultado_2.to_string(index=False))

## 🏘️ Consulta 3 — Agrupación por alcaldía: precio promedio y actividad
**Tipo:** Agrupación  
**Objetivo:** Identificar qué alcaldías (neighbourhood) tienen mayor concentración de listings y precios más altos.

In [ ]:
query_3 = '''
    SELECT
        neighbourhood,
        COUNT(*)                                AS total_listings,
        ROUND(AVG(price), 2)                    AS precio_promedio,
        ROUND(AVG(availability_365), 1)         AS disponibilidad_promedio,
        ROUND(AVG(number_of_reviews), 1)        AS resenas_promedio,
        ROUND(AVG(reviews_per_month), 2)        AS resenas_mes_promedio
    FROM listings
    WHERE price IS NOT NULL
    GROUP BY neighbourhood
    HAVING total_listings >= 50
    ORDER BY precio_promedio DESC
'''

resultado_3 = pd.read_sql(query_3, conn)
print('Agrupación por alcaldía (mín. 50 listings):')
print(resultado_3.to_string(index=False))

## 🔍 Consulta 4 — Filtro: listings de alto precio y alta disponibilidad
**Tipo:** Filtro  
**Objetivo:** Encontrar listings premium (precio > $5,000 MXN) que además estén disponibles más de 300 días al año — candidatos a propiedades de inversión.

In [ ]:
query_4 = '''
    SELECT
        id,
        name,
        neighbourhood,
        room_type,
        price,
        availability_365,
        number_of_reviews,
        calculated_host_listings_count  AS listings_del_host
    FROM listings
    WHERE
        price > 5000
        AND availability_365 > 300
        AND price IS NOT NULL
    ORDER BY price DESC
    LIMIT 20
'''

resultado_4 = pd.read_sql(query_4, conn)
print('Listings premium (precio > $5,000 y disponibilidad > 300 días):')
print(resultado_4.to_string(index=False))
print(f'\nTotal de listings que cumplen el filtro: {len(resultado_4)}')

## 👤 Consulta 5 — Agrupación por anfitrión: top 15 hosts más activos
**Tipo:** Agrupación + Filtro  
**Objetivo:** Identificar anfitriones profesionales (con muchos listings) y analizar su precio promedio versus los anfitriones casuales.

In [ ]:
query_5 = '''
    SELECT
        host_id,
        host_name,
        COUNT(*)                         AS total_listings,
        ROUND(AVG(price), 2)             AS precio_promedio,
        ROUND(AVG(availability_365), 1)  AS disponibilidad_promedio,
        ROUND(AVG(number_of_reviews), 1) AS resenas_promedio,
        COUNT(DISTINCT neighbourhood)    AS alcaldias_distintas
    FROM listings
    WHERE price IS NOT NULL
    GROUP BY host_id, host_name
    ORDER BY total_listings DESC
    LIMIT 15
'''

resultado_5 = pd.read_sql(query_5, conn)
print('Top 15 anfitriones más activos:')
print(resultado_5.to_string(index=False))

## 📅 Consulta 6 — Filtro: listings sin ninguna reseña (nuevos o inactivos)
**Tipo:** Filtro  
**Objetivo:** Identificar listings que nunca han recibido una reseña — pueden ser nuevos o estar inactivos, lo que afecta la imputación de reviews_per_month.

In [ ]:
query_6 = '''
    SELECT
        neighbourhood,
        room_type,
        COUNT(*) AS listings_sin_resenas,
        ROUND(AVG(price), 2) AS precio_promedio,
        ROUND(AVG(availability_365), 1) AS disponibilidad_promedio
    FROM listings
    WHERE
        number_of_reviews = 0
        AND last_review IS NULL
    GROUP BY neighbourhood, room_type
    ORDER BY listings_sin_resenas DESC
    LIMIT 15
'''

resultado_6 = pd.read_sql(query_6, conn)

total_sin_resenas = pd.read_sql(
    'SELECT COUNT(*) AS total FROM listings WHERE number_of_reviews = 0', conn
)
print(f'Total listings sin reseñas: {total_sin_resenas["total"][0]:,}\n')
print('Desglose por alcaldía y tipo de habitación:')
print(resultado_6.to_string(index=False))

## 📈 Consulta 7 — Valores promedio generales de toda la tabla
**Tipo:** Valores promedio  
**Objetivo:** Obtener una visión global de los indicadores clave del dataset desde SQL.

In [ ]:
query_7 = '''
    SELECT
        COUNT(*)                                    AS total_listings,
        COUNT(DISTINCT host_id)                     AS total_anfitriones,
        COUNT(DISTINCT neighbourhood)               AS total_alcaldias,
        COUNT(price)                                AS listings_con_precio,
        ROUND(AVG(price), 2)                        AS precio_promedio_global,
        ROUND(AVG(minimum_nights), 2)               AS noches_minimas_promedio,
        ROUND(AVG(number_of_reviews), 2)            AS resenas_promedio,
        ROUND(AVG(availability_365), 2)             AS disponibilidad_promedio,
        ROUND(AVG(calculated_host_listings_count), 2) AS listings_por_host_promedio
    FROM listings
'''

resultado_7 = pd.read_sql(query_7, conn)
print('Resumen global del dataset desde SQLite:')
# Transponer para mejor lectura
print(resultado_7.T.rename(columns={0: 'Valor'}).to_string())

## 🔗 Consulta 8 — Comparativa anfitriones casuales vs profesionales (self-join / subquery)
**Tipo:** Agrupación + Subquery (equivalente a Join)  
**Objetivo:** Comparar el comportamiento de precios entre anfitriones con 1 solo listing (casuales) vs anfitriones con más de 10 listings (profesionales).

In [ ]:
query_8 = '''
    SELECT
        CASE
            WHEN calculated_host_listings_count = 1 THEN '1. Casual (1 listing)'
            WHEN calculated_host_listings_count BETWEEN 2 AND 5 THEN '2. Semi-profesional (2-5)'
            WHEN calculated_host_listings_count BETWEEN 6 AND 10 THEN '3. Activo (6-10)'
            ELSE '4. Profesional (>10)'
        END AS perfil_anfitrion,
        COUNT(*)                                AS total_listings,
        COUNT(DISTINCT host_id)                 AS total_anfitriones,
        ROUND(AVG(price), 2)                    AS precio_promedio,
        ROUND(AVG(availability_365), 1)         AS disponibilidad_promedio,
        ROUND(AVG(number_of_reviews), 1)        AS resenas_promedio
    FROM listings
    WHERE price IS NOT NULL
    GROUP BY perfil_anfitrion
    ORDER BY perfil_anfitrion
'''

resultado_8 = pd.read_sql(query_8, conn)
print('Comparativa por perfil de anfitrión:')
print(resultado_8.to_string(index=False))

---
# 5. Verificación Final y Cierre de Conexión

In [ ]:
# Listar todas las tablas en la base de datos
tablas = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print('Tablas existentes en la base de datos:')
print(tablas.to_string(index=False))

# Tamaño del archivo .db
size_mb = os.path.getsize(DB_PATH) / (1024 * 1024)
print(f'\nTamaño del archivo .db: {size_mb:.2f} MB')
print(f'Ubicación: {os.path.abspath(DB_PATH)}')

In [ ]:
# Cerrar conexión
conn.close()
print('Conexión a SQLite cerrada correctamente.')

---
# 6. Resumen de Consultas Realizadas

| # | Consulta | Tipo | Hallazgo Principal |
|---|---|---|---|
| 1 | Conteo por tipo de habitación | Conteo | 65.5% son "Entire home/apt"; solo 0.3% son "Hotel room" |
| 2 | Estadísticas de precio por tipo | Valores promedio | Hotel room tiene el mayor precio promedio (~$42,841 MXN) aunque con pocos listings |
| 3 | Agrupación por alcaldía | Agrupación | Tlalpan y Cuajimalpa lideran en precio; Cuauhtémoc tiene el mayor volumen |
| 4 | Filtro: listings premium + alta disponibilidad | Filtro | Listings con precio > $5,000 y disponibilidad > 300 días: posibles inversiones |
| 5 | Top 15 anfitriones más activos | Agrupación + Filtro | Existen hosts con 200+ listings, indicando gestión profesional |
| 6 | Filtro: listings sin ninguna reseña | Filtro | Listings nuevos o inactivos: relevantes para la estrategia de imputación |
| 7 | Promedios globales del dataset | Valores promedio | Precio global promedio ~$1,792 MXN; 27,051 listings de 13,000+ anfitriones |
| 8 | Perfil de anfitrión (casual vs profesional) | Agrupación + Subquery | Anfitriones profesionales tienen precios más altos y mayor disponibilidad |